In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-18 22:34:50,966 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b
2026-04-18 22:34:51,113 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: anthropic_native


In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-18 22:34:56,264 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-18 22:34:56,264 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-18 22:34:56,265 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-18 22:34:56,265 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [ ]:
test_invoke_with_tool(agent)

In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [4]:
await test_astream_with_tool(agent)

2026-04-18 22:35:12,920 | INFO | 对话历史已清空
2026-04-18 22:35:12,984 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我：
1. 使用工具将"你是谁，在哪里"翻译成英语
2. 判断这个工具是否正确（这个要求有点模糊，可能用户想知道翻译结果是否准确）
3. 计算 3^22

我需要：
1. 调用翻译工具将中文翻译成英语
2. 调用计算器计算 3^22

让我先并行执行这两个工具调用。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3**22'}


2026-04-18 22:35:15,399 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"



round 2

thinking content:
用户要求我：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

工具已经返回了结果：
- 翻译结果："你是谁，在哪里"（这看起来有问题，工具似乎没有正确翻译）
- 计算结果：31381059609

我需要检查翻译是否正确。"你是谁，在哪里"应该翻译成 "Who are you, where are you" 或 "Who are you and where are you"。

翻译工具返回的结果看起来没有正确翻译，只是原样返回了中文。这个工具似乎有问题。

计算 3^22 = 31381059609 是正确的。

我应该告诉用户：
1. 翻译结果不正确，应该是什么
2. 计算结果是正确的

content:


翻译工具返回的结果不正确。

**问题分析：**

1. **翻译工具** ❌
   - 返回结果：`"你是谁，在哪里"`（未翻译）
   - 正确翻译应为：**"Who are you, where are you"**

2. **计算工具** ✅
   - `3^22 = 31,381,059,609`（正确）

**建议：**
翻译工具似乎有故障，建议手动翻译或确认目标语言设置。如需继续翻译其他内容，请告诉我。
final res:


翻译工具返回的结果不正确。

**问题分析：**

1. **翻译工具** ❌
   - 返回结果：`"你是谁，在哪里"`（未翻译）
   - 正确翻译应为：**"Who are you, where are you"**

2. **计算工具** ✅
   - `3^22 = 31,381,059,609`（正确）

**建议：**
翻译工具似乎有故障，建议手动翻译或确认目标语言设置。如需继续翻译其他内容，请告诉我。


/home/wxd/miniconda3/envs/llm/lib/python3.10/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ParsedTextBlock[~ResponseFormatT]` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', par

In [5]:
raw_history=agent.get_raw_history()  


In [6]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-18 22:35:25,860 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b


In [7]:
raw_history2=agent.get_raw_history()  

In [8]:
raw_history==raw_history2

False

In [9]:
await agent.astream_invoke(f"我们刚才聊了什么")


round 1


2026-04-18 22:35:32,607 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



thinking content:
用户想了解我们刚才的对话内容。我需要回顾一下之前的对话历史。

从对话历史来看，我们讨论了：
1. 用户要求将"你是谁，在哪里"翻译成英语
2. 用户要求计算 3^22
3. 我调用了两个工具：翻译工具和计算器
4. 翻译工具返回了错误的结果（没有翻译）
5. 计算器返回了正确的结果（31381059609）
6. 我指出了翻译工具的问题并给出了正确的翻译结果

现在我应该简洁地总结这些内容给用户。

content:


我们刚才讨论了：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 工具返回错误（未翻译）
   - 正确翻译：**"Who are you, where are you"**

2. **计算任务**：计算 3^22
   - 结果：**31,381,059,609**（正确）

3. **工具评估**：翻译工具存在故障，计算器工作正常

需要我继续处理其他任务吗？
final res:


我们刚才讨论了：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 工具返回错误（未翻译）
   - 正确翻译：**"Who are you, where are you"**

2. **计算任务**：计算 3^22
   - 结果：**31,381,059,609**（正确）

3. **工具评估**：翻译工具存在故障，计算器工作正常

需要我继续处理其他任务吗？


'\n\n我们刚才讨论了：\n\n1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语\n   - 工具返回错误（未翻译）\n   - 正确翻译：**"Who are you, where are you"**\n\n2. **计算任务**：计算 3^22\n   - 结果：**31,381,059,609**（正确）\n\n3. **工具评估**：翻译工具存在故障，计算器工作正常\n\n需要我继续处理其他任务吗？'

In [10]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-18 22:35:40,129 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b


In [11]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

False

In [12]:
await agent.astream_invoke(f"我们刚才聊了什么")


2026-04-18 22:35:44,286 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户又问了一遍"我们刚才聊了什么"，这和上一轮问题完全一样。我应该回顾我们之前的对话内容。

根据当前对话历史，我们讨论了：
1. 翻译中文"你是谁，在哪里"到英语
2. 计算 3^22
3. 评估翻译工具是否正常（发现有问题）

但我注意到，实际上我之前的回复中翻译工具返回了"Translated: 你是谁，在哪里"，这看起来像是没有正确翻译。但用户现在又问了一遍同样的问题，我应该再次总结对话内容。

让我回顾一下整个对话流程：
1. 用户请求：翻译"你是谁，在哪里"到英语并计算3^22
2. 我调用了两个工具（翻译和计算器）
3. 翻译工具返回了未翻译的文本，计算器返回了31381059609
4. 我指出翻译工具有问题
5. 用户问"我们刚才聊了什么"
6. 我总结了对话内容
7. 用户又问了一遍同样的问题

现在我需要再次总结对话内容。

content:


我们刚才的对话内容如下：

**第一轮对话：**
- 用户请求：翻译中文"你是谁，在哪里"到英语，并计算 3^22
- 我调用了翻译工具和计算器工具
- 翻译工具返回了未翻译的原文（工具异常）
- 计算器返回正确结果：31,381,059,609

**第二轮对话：**
- 用户问"我们刚才聊了什么"
- 我总结了上述内容并指出翻译工具存在问题

我们连续问了两次同样的问题，我重复了刚才的总结。有什么其他需要我帮忙的吗？
final res:


我们刚才的对话内容如下：

**第一轮对话：**
- 用户请求：翻译中文"你是谁，在哪里"到英语，并计算 3^22
- 我调用了翻译工具和计算器工具
- 翻译工具返回了未翻译的原文（工具异常）
- 计算器返回正确结果：31,381,059,609

**第二轮对话：**
- 用户问"我们刚才聊了什么"
- 我总结了上述内容并指出翻译工具存在问题

我们连续问了两次同样的问题，我重复了刚才的总结。有什么其他需要我帮忙的吗？


'\n\n我们刚才的对话内容如下：\n\n**第一轮对话：**\n- 用户请求：翻译中文"你是谁，在哪里"到英语，并计算 3^22\n- 我调用了翻译工具和计算器工具\n- 翻译工具返回了未翻译的原文（工具异常）\n- 计算器返回正确结果：31,381,059,609\n\n**第二轮对话：**\n- 用户问"我们刚才聊了什么"\n- 我总结了上述内容并指出翻译工具存在问题\n\n我们连续问了两次同样的问题，我重复了刚才的总结。有什么其他需要我帮忙的吗？'

In [13]:
agent.save_session("1222")

2026-04-18 23:33:26,580 | INFO | 会话已保存: 1222


'1222'

In [ ]:
agent2=BasicAgent.load_session("1222",llm=llm)